# Step 0: Notebook context

## 1-url-language-country-analyzer.ipynb

This notebook analyzes a filtered dataset (Google Alerts - BioArt) from the spreadsheet sheet `Bio1Year`, extracting content, language, and country information from listed URLs.

> **Note:** This dataset has already been deduplicated in `0-clean-duplicates.ipynb`.


# Step 1: Install and import dependencies

In [1]:
!pip install --upgrade google-api-python-client google-auth google-auth-oauthlib python-dotenv beautifulsoup4 langdetect pandas requests

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 20.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for langdetect: filename=langdetect-1.0.9-py3-none-any.whl size=993222 sha256=c11116ea315d12737ca396b229ff20a551033b9059a10481a2b8a7713a6aaebd
  Stored in directory: /Users/linalopes/Library/Caches/pip/wheels/c1/67/88/e844b5b022812e15a52e4eaa38a1e709e99f06f6639d7e3ba7
Successfully built langdetect
  Attempting uninstall: beautifulsoup4
    Found existing installation: beautifulsoup4 4.13.3
    Uninstalling beautifulsoup4-4.13.3:
      Successfully uninstalled beautifulsoup4-4.13.3


In [2]:
import os
from dotenv import load_dotenv
from googleapiclient.discovery import build
from google.oauth2 import service_account

import requests
from bs4 import BeautifulSoup
from langdetect import detect
from langdetect.lang_detect_exception import LangDetectException
from urllib.parse import urlparse

import pandas as pd
import logging


# Step 2: Load credentials and connect to Google Sheets

In [6]:
load_dotenv()

SERVICE_ACCOUNT_FILE = os.getenv('SERVICE_ACCOUNT_FILE')
SPREADSHEET_ID = os.getenv('SPREADSHEET_ID')
SHEET_NAME = 'BioArt1Year'  # dynamic target sheet

SCOPES = ['https://www.googleapis.com/auth/spreadsheets']

creds = service_account.Credentials.from_service_account_file(
    SERVICE_ACCOUNT_FILE, scopes=SCOPES)
service = build('sheets', 'v4', credentials=creds)
sheet = service.spreadsheets()


# Step 3: Read and filter `bioart` rows from `Sheet0`

In [15]:
# Read full data from Sheet0 (excluding header)
result = sheet.values().get(
    spreadsheetId=SPREADSHEET_ID,
    range='Sheet0!A2:Z'  # Adjust range if you know exactly how many columns
).execute()

rows = result.get('values', [])

# Define expected column names
columns = ['wordAlert', 'link', 'date', 'source', 'title', 'description']

# Normalize rows to same length (fill missing values)
normalized_rows = [row + [''] * (len(columns) - len(row)) for row in rows]

# Convert to DataFrame
df = pd.DataFrame(normalized_rows, columns=columns)

# Filter only rows where wordAlert is 'bioart' (case-insensitive)
df_bioart = df[df['wordAlert'].str.strip().str.lower() == 'bioart'].copy()

print(f"🔍 Found {len(df_bioart)} rows with wordAlert='bioart'.")
df_bioart.head()


🔍 Found 461 rows with wordAlert='bioart'.


,wordAlert,link,date,source,title,description
35,bioart,https://www.nedeljnik.rs/bio-avaking-spoj-umet...,2024-03-28,nedeljnik.rs,Bio Awaking: Spoj umetnosti i nauke za održivu...,"... bioart. Serijom događaja, studijskih poset..."
36,bioart,https://www.sn.at/salzburg/chronik/fuer-seeham...,2024-03-15,sn.at,Für Seehamer Röster ist Kaffee eine Lebenseins...,"Beim ersten ""Kaffeekränzchen"" im BioArt-Campus..."
224,bioart,https://nauka.telegraf.rs/naucne-zanimljivosti...,2024-05-28,nauka.telegraf.rs,Šta se dešava kada umetnici uđu u naučne labor...,To smo imali prilike da vidimo na Bio Art Foru...
225,bioart,https://www.corfupress.com/2024/%CE%B5%CE%B9%C...,2024-05-28,corfupress.com,Εγκαίνια της έκθεσης “Το Μεταλλαξιογόνο Μέλλον...,"... Bioart Ethical Advisory Kommission), World..."
226,bioart,https://www.editoraroncarati.com.br/v2/Artigos...,2024-05-24,editoraroncarati.com.br,ALERTAS ANVISA EM 24.05.2024 | Artigos e Notíc...,... Bioart. 2- Seguir as orientações de como c...


# Step 4: Analyze URLs to extract language, country, and text

For each URL in the filtered dataset (`wordAlert == "bioart"`), we attempt to fetch the page content and extract:
- The main textual content (truncated to 25,000 characters).
- The language detected using the `langdetect` library.
- The country (if available) from metadata tags in the HTML.

If a URL cannot be fetched or parsed, we label the result as `"BLOCKED"` to maintain consistent row count in the spreadsheet.


In [16]:
def get_country_from_metadata(soup):
    meta_tags = [
        {'name': 'geo.country'},
        {'property': 'og:country-name'},
        {'name': 'country'},
        {'name': 'dcterms.coverage'},
        {'name': 'ICBM'},
        {'name': 'geo.position'},
        {'name': 'geo.placename'},
    ]
    for tag_attrs in meta_tags:
        meta = soup.find('meta', attrs=tag_attrs)
        if meta and 'content' in meta.attrs:
            return meta['content'].strip()
    return 'Unknown'

def analyze_url(url):
    if not url:
        return "NO_URL", "NO_COUNTRY", "NO_TEXT"

    parsed = urlparse(url)
    if not parsed.scheme:
        url = 'http://' + url

    try:
        headers = {'User-Agent': 'Mozilla/5.0'}
        response = requests.get(url, headers=headers, timeout=10)
        response.raise_for_status()

        soup = BeautifulSoup(response.content, 'html.parser')
        text = soup.get_text(separator=' ', strip=True)
        text_snippet = text[:25000] if text else 'NO_TEXT'

        try:
            language = detect(text_snippet)
        except LangDetectException:
            language = 'UNKNOWN'

        country = get_country_from_metadata(soup)

        return language, country, text_snippet

    except Exception as e:
        logging.warning(f"Error fetching or parsing {url}: {e}")
        return "BLOCKED", "BLOCKED", "BLOCKED"

# Step 5: Apply the analysis function to filtered rows

We apply `analyze_url()` to each row in the filtered dataset.  
To preserve the row positioning in `Sheet0`, we track the actual index positions of each `bioart` entry.


In [17]:
# Get index positions of the rows in Sheet0 where wordAlert == 'bioart'
bioart_indices_in_sheet = df.index[df['wordAlert'].str.strip().str.lower() == 'bioart'].tolist()

# Analyze each URL and collect results
results = df_bioart['link'].apply(analyze_url)
df_bioart[['detected-language', 'detected-country', 'detected-text']] = pd.DataFrame(results.tolist(), index=df_bioart.index)

# Confirm output
df_bioart[['link', 'detected-language', 'detected-country']].head()

,link,detected-language,detected-country
35,https://www.nedeljnik.rs/bio-avaking-spoj-umet...,hr,Unknown
36,https://www.sn.at/salzburg/chronik/fuer-seeham...,de,Unknown
224,https://nauka.telegraf.rs/naucne-zanimljivosti...,hr,Unknown
225,https://www.corfupress.com/2024/%CE%B5%CE%B9%C...,el,Unknown
226,https://www.editoraroncarati.com.br/v2/Artigos...,pt,Unknown


# Step 6: Efficient batch update of columns G–I in `Sheet0`

We split the batch update into smaller chunks (e.g. 200 rows at a time), to stay within the Google Sheets API limits.

We:
- Identify the range of rows to be updated.
- Fill in placeholder rows (`["", "", ""]`) for non-`bioart` entries to preserve row alignment.
- Write the entire block in a single `values().update()` request.


In [21]:
import math

def chunked_write_to_sheet0(starting_row, data, chunk_size=200):
    total_rows = len(data)
    num_chunks = math.ceil(total_rows / chunk_size)

    for i in range(num_chunks):
        chunk_start_index = i * chunk_size
        chunk_end_index = min((i + 1) * chunk_size, total_rows)

        # Prepare the chunk data
        chunk_data = data[chunk_start_index:chunk_end_index]

        # Calculate the actual row numbers in the sheet
        sheet_start_row = starting_row + chunk_start_index
        sheet_end_row = sheet_start_row + len(chunk_data) - 1

        # Build the range string
        range_str = f'Sheet0!G{sheet_start_row}:I{sheet_end_row}'

        # Write to sheet
        service.spreadsheets().values().update(
            spreadsheetId=SPREADSHEET_ID,
            range=range_str,
            valueInputOption='RAW',
            body={'values': chunk_data}
        ).execute()

        print(f"✅ Wrote rows {sheet_start_row}–{sheet_end_row}")

# Chama a função com os dados filtrados
chunked_write_to_sheet0(first_row, full_update_rows, chunk_size=200)


✅ Wrote rows 37–236
✅ Wrote rows 237–436
✅ Wrote rows 437–636
✅ Wrote rows 637–836
✅ Wrote rows 837–1036
✅ Wrote rows 1037–1236
✅ Wrote rows 1237–1436
✅ Wrote rows 1437–1636
✅ Wrote rows 1637–1836
✅ Wrote rows 1837–2036
✅ Wrote rows 2037–2236
✅ Wrote rows 2237–2436
✅ Wrote rows 2437–2636
✅ Wrote rows 2637–2836
✅ Wrote rows 2837–3036
✅ Wrote rows 3037–3236
✅ Wrote rows 3237–3436
✅ Wrote rows 3437–3636
✅ Wrote rows 3637–3836
✅ Wrote rows 3837–4036
✅ Wrote rows 4037–4236
✅ Wrote rows 4237–4436
✅ Wrote rows 4437–4626


# Step 7 (Optional): Summary of processing results

We create a quick summary showing how many `bioart` entries were successfully processed, and how many failed due to blocked access or invalid URLs. This helps estimate data loss or scraping limitations.


In [22]:
# Mark status based on language column
df_bioart['status'] = df_bioart['detected-language'].apply(
    lambda x: 'OK' if x not in ['BLOCKED', 'NO_URL'] else 'ERROR'
)

# Show counts
status_counts = df_bioart['status'].value_counts(normalize=True).round(3) * 100

# Display as percentages
print("📊 Processing summary (% of total):")
print(status_counts)


📊 Processing summary (% of total):
status
OK       98.0
ERROR     2.0
Name: proportion, dtype: float64
